# YOLOv8 Training with COCO Annotations (Train/Val Split)

이 노트북은 COCO 형식의 annotation을 사용하여 YOLOv8 모델을 학습하고 성능을 평가합니다.

**주요 특징**: 
- annotations 폴더의 모든 파일을 합쳐서 하나의 annotation으로 병합
- Train/Val 셋을 8:2 비율로 분리
- 단일 images/labels 폴더 구조 사용
- 테스트 성능 평가 포함

In [62]:
import os
import json
import yaml
import logging
from pathlib import Path
import numpy as np
from datetime import datetime
import glob
from sklearn.model_selection import train_test_split

# 작업 디렉토리 설정
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir) if 'notebooks' in notebook_dir else notebook_dir
os.chdir(project_root)

print(f"프로젝트 루트: {os.getcwd()}")

# YOLOv8 라이브러리
try:
    from ultralytics import YOLO
    import pycocotools.coco
    print("✓ YOLOv8 및 COCO 라이브러리 로드 완료")
except ImportError as e:
    print(f"✗ 라이브러리 로드 실패: {e}")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

프로젝트 루트: /data/th/KETI/Grounded-SAM-2
✓ YOLOv8 및 COCO 라이브러리 로드 완료


In [63]:
# 디렉토리 확인
required_dirs = ['images', 'annotations']
for dir_name in required_dirs:
    if os.path.isdir(dir_name):
        file_count = len(os.listdir(dir_name))
        print(f"✓ {dir_name}/ ({file_count}개 파일)")
    else:
        print(f"✗ {dir_name}/ (찾을 수 없음)")

✓ images/ (201개 파일)
✓ annotations/ (145개 파일)


In [64]:
# COCO annotation 파일 찾기 (merged_annotations.json 제외)
def find_annotation_files(annotations_dir="annotations"):
    all_files = glob.glob(os.path.join(annotations_dir, "*_coco.json"))
    # merged 또는 backup이 포함된 파일 제외
    annotation_files = sorted([f for f in all_files if 'merged' not in os.path.basename(f).lower() and 'backup' not in os.path.basename(f).lower()])
    logging.info(f"찾은 annotation 파일: {len(annotation_files)}개")
    return annotation_files

annotation_files = find_annotation_files()
print(f"총 {len(annotation_files)}개의 annotation 파일 발견")

2025-11-21 16:23:49,851 - INFO - 찾은 annotation 파일: 143개


총 143개의 annotation 파일 발견


In [65]:
# 여러 COCO annotation 파일 병합
def merge_coco_annotations(annotation_files):
    merged_data = {
        "info": {"description": "Merged COCO Annotations", "version": "1.0", "year": datetime.now().year},
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }
    
    category_map = {}
    next_image_id = 1
    next_annotation_id = 1
    next_category_id = 1
    
    for file_path in annotation_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 카테고리 병합
            file_category_map = {}
            for category in data.get("categories", []):
                cat_name = category.get("name")
                old_cat_id = category.get("id")
                
                existing_cat = next((c for c in merged_data["categories"] if c.get("name") == cat_name), None)
                if existing_cat:
                    file_category_map[old_cat_id] = existing_cat["id"]
                else:
                    new_cat_id = next_category_id
                    next_category_id += 1
                    file_category_map[old_cat_id] = new_cat_id
                    merged_data["categories"].append({"id": new_cat_id, "name": cat_name, "supercategory": "thing"})
            
            # 이미지 및 annotation 병합
            file_image_map = {}
            for image in data.get("images", []):
                old_image_id = image.get("id")
                new_image_id = next_image_id
                next_image_id += 1
                file_image_map[old_image_id] = new_image_id
                
                new_image = image.copy()
                new_image["id"] = new_image_id
                merged_data["images"].append(new_image)
            
            for annotation in data.get("annotations", []):
                new_annotation = annotation.copy()
                new_annotation["id"] = next_annotation_id
                next_annotation_id += 1
                new_annotation["image_id"] = file_image_map.get(annotation.get("image_id"))
                new_annotation["category_id"] = file_category_map.get(annotation.get("category_id"))
                merged_data["annotations"].append(new_annotation)
        
        except Exception as e:
            logging.error(f"파일 병합 실패 {file_path}: {e}")
            continue
    
    logging.info(f"병합 완료: {len(merged_data['images'])}개 이미지, {len(merged_data['annotations'])}개 annotation")
    return merged_data

merged_coco_data = merge_coco_annotations(annotation_files)
print(f"✓ 병합 완료: {len(merged_coco_data['images'])}개 이미지, {len(merged_coco_data['categories'])}개 카테고리")

2025-11-21 16:23:51,191 - INFO - 병합 완료: 143개 이미지, 429개 annotation


✓ 병합 완료: 143개 이미지, 3개 카테고리


In [66]:
# 병합된 annotation 저장
merged_file = "annotations/merged_annotations.json"
with open(merged_file, 'w', encoding='utf-8') as f:
    json.dump(merged_coco_data, f, ensure_ascii=False, indent=2)

print(f"✓ 병합된 annotation 저장: {merged_file}")

✓ 병합된 annotation 저장: annotations/merged_annotations.json


In [67]:
# COCO 형식을 YOLO 형식으로 변환
def convert_bbox_to_yolo(image_width, image_height, bbox):
    """COCO bbox를 YOLO 형식으로 변환"""
    x, y, w, h = bbox
    x_center = (x + w / 2) / image_width
    y_center = (y + h / 2) / image_height
    w_norm = w / image_width
    h_norm = h / image_height
    return x_center, y_center, w_norm, h_norm

def coco_to_yolo(merged_coco_data, output_images_dir="dataset/images", output_labels_dir="dataset/labels", images_dir="images"):
    """병합된 COCO를 YOLO 형식으로 변환"""
    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)
    
    image_map = {img['id']: img for img in merged_coco_data.get('images', [])}
    used_images = []
    
    for image_id, img_info in image_map.items():
        img_filename = img_info['file_name']
        img_path = os.path.join(images_dir, img_filename)
        
        if not os.path.exists(img_path):
            logging.warning(f"이미지를 찾을 수 없음: {img_filename}")
            continue
        
        # 심볼릭 링크 생성
        link_path = os.path.join(output_images_dir, img_filename)
        if not os.path.exists(link_path):
            os.symlink(os.path.abspath(img_path), link_path)
        
        img_width, img_height = img_info['width'], img_info['height']
        
        # 라벨 파일 생성
        image_annotations = [ann for ann in merged_coco_data.get('annotations', [])
                           if ann.get('image_id') == image_id]
        
        label_file_path = os.path.join(output_labels_dir, f"{img_filename.split('.')[0]}.txt")
        with open(label_file_path, 'w') as label_file:
            for ann in image_annotations:
                if 'bbox' in ann and ann.get('area', 0) > 0:
                    bbox = convert_bbox_to_yolo(img_width, img_height, ann['bbox'])
                    class_id = ann['category_id'] - 1
                    label_file.write(f"{class_id} {' '.join(map(str, bbox))}\n")
        
        used_images.append(img_filename)
    
    logging.info(f"YOLO 변환 완료: {len(used_images)}개 이미지")
    return used_images, merged_coco_data.get('categories', [])

used_images, categories = coco_to_yolo(merged_coco_data)
print(f"✓ YOLO 형식 변환 완료: {len(used_images)}개 이미지")

2025-11-21 16:23:53,394 - INFO - YOLO 변환 완료: 143개 이미지


✓ YOLO 형식 변환 완료: 143개 이미지


In [68]:
# Train/Val 셋 분리 (80:20)
train_images, val_images = train_test_split(used_images, test_size=0.2, random_state=42)

print(f"✓ Train/Val 분리 완료")
print(f"  - Train: {len(train_images)}개 이미지")
print(f"  - Val: {len(val_images)}개 이미지")

# Train/Val 리스트를 텍스트 파일로 저장
os.makedirs("dataset", exist_ok=True)

with open("dataset/train.txt", 'w') as f:
    for img in train_images:
        f.write(os.path.abspath(os.path.join("dataset/images", img)) + "\n")

with open("dataset/val.txt", 'w') as f:
    for img in val_images:
        f.write(os.path.abspath(os.path.join("dataset/images", img)) + "\n")

print(f"✓ Train/Val 리스트 파일 생성 완료")

✓ Train/Val 분리 완료
  - Train: 114개 이미지
  - Val: 29개 이미지
✓ Train/Val 리스트 파일 생성 완료


In [69]:
# data.yaml 생성
class_names = {cat['id'] - 1: cat['name'] for cat in categories}

data_yaml = {
    'path': os.path.abspath("dataset"),
    'train': os.path.abspath("dataset/train.txt"),
    'val': os.path.abspath("dataset/val.txt"),
    'nc': len(categories),
    'names': class_names
}

yaml_path = "dataset/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✓ data.yaml 생성 완료")
print(f"\n클래스 정보:")
for class_id, class_name in sorted(class_names.items()):
    print(f"  {class_id}: {class_name}")

✓ data.yaml 생성 완료

클래스 정보:
  0: DOOR
  1: JOG
  2: MEMORY


In [70]:
# YOLOv8 모델 학습
job_id = datetime.now().strftime('%Y%m%d_%H%M%S')

try:
    logging.info("YOLOv8 모델 초기화 및 학습 시작")
    model = YOLO('yolov8s.pt')
    
    results = model.train(
        data=yaml_path,
        epochs=10,
        batch=16,
        imgsz=640,
        project="trained_models",
        name=f"yolo_{job_id}",
        verbose=False
    )
    
    print(f"✓ 모델 학습 완료")
    
except Exception as e:
    print(f"✗ 학습 실패: {e}")
    import traceback
    traceback.print_exc()

2025-11-21 16:23:58,667 - INFO - YOLOv8 모델 초기화 및 학습 시작


New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.203 🚀 Python-3.11.0 torch-2.3.1+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo_20251121_162358, nbs=64, nms=F

In [71]:
# Val 셋 검증
try:
    logging.info("Val 셋 검증 시작")
    val_results = model.val()
    
    print(f"✓ Val 셋 검증 완료")
    
except Exception as e:
    print(f"✗ 검증 실패: {e}")

2025-11-21 16:24:25,354 - INFO - Val 셋 검증 시작


Ultralytics 8.3.203 🚀 Python-3.11.0 torch-2.3.1+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81051MiB)
Model summary (fused): 72 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1231.9±408.3 MB/s, size: 436.2 KB)
val: Scanning /data/th/KETI/Grounded-SAM-2/dataset/labels.cache... 29 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29/29 54.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.2it/s 0.9s1.2s
                   all         29         87      0.972      0.923      0.973      0.805
Speed: 1.4ms preprocess, 15.9ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /data/th/KETI/Grounded-SAM-2/trained_models/yolo_20251121_1623582
✓ Val 셋 검증 완료


In [72]:
# 성능 메트릭 출력
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("YOLOv8 학습 및 검증 결과")
print("="*60)

print(f"\n학습 설정:")
print(f"  - 총 이미지: {len(used_images)}개")
print(f"  - Train: {len(train_images)}개 ({len(train_images)/len(used_images)*100:.1f}%)")
print(f"  - Val: {len(val_images)}개 ({len(val_images)/len(used_images)*100:.1f}%)")
print(f"  - 에포크: 10")
print(f"  - 배치 크기: 16")

if 'val_results' in locals():
    print(f"\n검증 메트릭:")
    print(f"  - mAP@50: {val_results.box.map50:.4f}")
    print(f"  - mAP@50-95: {val_results.box.map:.4f}")
    print(f"  - Precision: {val_results.box.mp:.4f}")
    print(f"  - Recall: {val_results.box.mr:.4f}")
    
    metrics_data = {
        'mAP@50': val_results.box.map50,
        'mAP@50-95': val_results.box.map,
        'Precision': val_results.box.mp,
        'Recall': val_results.box.mr
    }
    
    # 메트릭 시각화
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics_names = list(metrics_data.keys())
    metrics_values = list(metrics_data.values())
    
    bars = ax.bar(metrics_names, metrics_values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('YOLOv8 Validation Metrics', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\n" + "="*60)


YOLOv8 학습 및 검증 결과

학습 설정:
  - 총 이미지: 143개
  - Train: 114개 (79.7%)
  - Val: 29개 (20.3%)
  - 에포크: 10
  - 배치 크기: 16

검증 메트릭:
  - mAP@50: 0.9728
  - mAP@50-95: 0.8047
  - Precision: 0.9722
  - Recall: 0.9231


<Figure size 1000x600 with 1 Axes>